# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shwetabh1013/flyrank-ml-internship-shwetabh/blob/main/work/notebooks/capstone.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

This notebook is the paper's backbone: every number and chart on the deployed page comes from here. It reads the results W06 (validation) and W07 (playbook) write to `work/outputs/`. Those CSVs are git-ignored, so on a fresh clone the first cell runs W06 and W07 for you (~6 minutes).

In [1]:
import json, subprocess, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

sys.path.append(str(Path("..").resolve()))
import refresh_lib as rl
warnings.filterwarnings("ignore")

OUT = Path("../outputs"); IMG = Path("../../docs/img")
IMG.mkdir(parents=True, exist_ok=True)

needed = {"w06_validation_audit.ipynb": OUT / "w06_split_compare.csv",
          "w07_action_playbook.ipynb": OUT / "w07_queue_top10_per_client.csv"}
for nb_name, out_file in needed.items():
    if not out_file.exists():
        print(f"{out_file.name} missing -> executing {nb_name}")
        subprocess.run(["jupyter", "nbconvert", "--to", "notebook", "--execute", "--inplace",
                        "--ExecutePreprocessor.timeout=1800", nb_name], check=True, capture_output=True)

df = rl.load_data()
folds = pd.read_csv(OUT / "w06_fold_results.csv")
queue = pd.read_csv(OUT / "w07_queue_top10_per_client.csv")
coefs = pd.read_csv(OUT / "w07_model_coefficients.csv", index_col=0)["coefficient"]
w07 = json.loads((OUT / "w07_summary.json").read_text())
print("loaded:", len(folds), "fold rows |", len(queue), "queue rows")

loaded: 175 fold rows | 280 queue rows


## 1. Question

**Which pages should an editor review first because they are about to lose search visibility — and does a trained model pick them better than the simple rules we started with?**

Decision supported: each month, an editor for one client site chooses ~10 pages to review for a refresh. Today that choice is gut feel (oldest pages, or lowest traffic). A wrong pick costs an editor hour on a page that was fine; a miss lets a page keep sliding.

Unit: one page. Output: a ranked review queue per client, with reason codes. Primary metric: precision@20 (how many of the top picks really decline), always next to the base rate.

## 2. Data

- **Release:** the FlyRank ML Internship starter slice, `content_refresh_anonymized.csv` — 30,000 pages, 32 pseudonymised clients, one 90-day snapshot. (The full Hugging Face warehouse was used in W03 for the data contract; the modelling stays on the starter slice so every number is reproducible without a token.)
- **Windows:** impressions/clicks/sessions split into days 1–30, 31–60, and 90-day totals; days 61–90 derived by subtraction.
- **Excluded and why:**
  - pages with < 30 impressions in days 31–60 (the label's denominator) — a 20% "decline" on 3 impressions is noise;
  - `trend_direction`, `trend_pct` — the label's source;
  - any 90-day aggregate (CTR, position, engagement) and update dates — they overlap the label's window (see Methodology);
  - `client_id`, `content_id` — grouping only; `provider_used`, `model_used` — generation metadata, not page signals.
- Nothing client-identifying appears in this notebook or the paper; ids are pseudonyms.

In [2]:
m = rl.measurable(df)
data_summary = pd.Series({
    "pages (all)": len(df), "clients (all)": df["client_id"].nunique(),
    "pages (measurable)": len(m), "clients (measurable)": m["client_id"].nunique(),
    "declining rate (all)": round(df["y"].mean(), 3),
    "declining rate (measurable)": round(m["y"].mean(), 3),
    "pages updated inside label window": f"{(df['days_since_last_update'] < 30).mean():.1%}",
})
data_summary

pages (all)                          30000
clients (all)                           32
pages (measurable)                   21578
clients (measurable)                    31
declining rate (all)                 0.542
declining rate (measurable)          0.615
pages updated inside label window    68.3%
dtype: object

## 3. Methodology

- **Label:** `is_declining_label` = impressions in days 1–30 fell more than 20% vs days 31–60 (`trend_direction == "down"`), on measurable pages only.
- **Assumption — the decision is made before the outcome.** Features may only use days 31–90 and static metadata ("as of T-30"). For the live queue the same features are shifted to days 1–60, forecasting the next 30 days.
- **Features (23):** log impressions/clicks/sessions for each of the two pre-label months, impression momentum, 60-day CTR, content age, word count, keyword volume/competition/CPC with `has_` flags (missingness follows content type), content type, intent.
- **Baselines:** random ranking (= base rate), staleness only (W02), the W04 rule (stale AND CTR below half its tier benchmark, benchmark fit on train clients only), impression momentum.
- **Models:** logistic regression and random forest; selection rule fixed in advance = higher mean precision@20.
- **Validation:** GroupKFold by client, 5 folds × 5 shuffles = 25 held-out client groups; same folds for every candidate; an assertion checks no client is on both sides.
- **Leakage checks:** banned-column assertion; window-overlap audit (found and fixed — W05's features overlapped the label window); a planted leak (`impressions_last_30d`) sends precision@20 to 1.0, confirming the harness detects leaks; random-split vs grouped-split gap reported.

## 4. Results (vs baseline)

Same 25 folds for every row. Mean ± std across folds.

In [3]:
order = ["random ranking", "staleness only", "W04 rule", "momentum",
         "RF, W05 features (overlapping windows)", "RF, honest features", "LR, honest features"]
agg = folds.groupby("model").agg(
    p20=("p20", "mean"), p20_sd=("p20", "std"), p50=("p50", "mean"),
    client_p10=("client_p10", "mean"), auc=("auc", "mean"), base_rate=("base_rate", "mean")).loc[order]
wide = folds.pivot_table(index=["seed", "fold"], columns="model", values="p20")
agg["beats_W04_folds"] = [(wide[mdl] >= wide["W04 rule"]).mean() for mdl in order]
results_table = agg.round(3)
results_table.to_csv(OUT / "capstone_results_table.csv")
results_table

,p20,p20_sd,p50,client_p10,auc,base_rate,beats_W04_folds
model,,,,,,,
random ranking,0.614,0.156,0.622,0.605,0.495,0.633,0.60
staleness only,0.702,0.166,0.702,0.617,0.489,0.633,0.72
W04 rule,0.582,0.138,0.586,0.618,0.522,0.633,1.00
momentum,0.626,0.138,0.652,0.612,0.481,0.633,0.64
"RF, W05 features (overlapping windows)",0.844,0.141,0.864,0.828,0.662,0.633,1.00
"RF, honest features",0.746,0.194,0.774,0.730,0.625,0.633,0.88
"LR, honest features",0.828,0.091,0.818,0.758,0.616,0.633,0.96


**Read:**
- The selected model (**LR, honest features**) lifts precision@20 from a **0.633 base rate to 0.828** (± 0.091) and matches or beats the W04 rule on 96% of folds.
- The **W04 rule (0.582) and staleness-only (0.702, AUC 0.49) don't rank declines** on unseen clients. Staleness is mostly two batch-update dates.
- The RF on W05 features scores higher (0.844, AUC 0.66), but only because its inputs overlap the outcome window — not deployable.
- Overall discrimination is modest (AUC ≈ 0.62): the lift lives at the **top of the queue**.

## 5. Limitations

- **Observational, one snapshot.** No refresh was randomised; nothing here says refreshing a page *causes* recovery.
- **Label is a proxy.** ">20% fewer impressions month over month" can be seasonality, a SERP layout change, or tracking — not just content decay.
- **Modest discrimination** (AUC ≈ 0.62). Use for top-10 review lists only.
- **High variance across client groups** (precision@20 std ≈ 0.09; base rates vary by ± 0.10). Some clients' queues will be much weaker than the average.
- **Age shift.** Scoring pages are 30 days older than training pages (PSI 0.93 on age); "younger = riskier" is the least-tested assumption.
- **No position, engagement or update-date signal** in the honest model — the CSV has no sub-window for them. The warehouse's daily table could add them.
- **Starter slice only** (30k pages, 32 clients, all ≥ 90 days old); not the full portfolio.
- **Model selection used the same CV** as evaluation (2 candidates, so optimism is small but non-zero); no untouched final holdout.

## 6. Ranked recommendations

The action playbook (from W07), in priority order:

In [4]:
print("Live queue:", w07["client_queues"], "client queues,", w07["queue_pages"], "pages")
print("Action mix:", w07["action_mix"])
print("Lead reason:", w07["lead_reason_mix"], "\n")
live = df[df["impressions_last_30d"] >= rl.MEASURABLE_FLOOR]
sample_client = live["client_id"].value_counts().index[0]   # largest client, same as W07
sample = queue[queue["client_id"] == sample_client][
    ["queue_rank", "content_id", "action", "reason_codes", "impressions_last_30d",
     "clicks_last_30d", "content_age_days"]]
sample.to_csv(OUT / "capstone_sample_queue.csv", index=False)
sample

Live queue: 29 client queues, 280 pages
Action mix: {'REVIEW_FIRST': 180, 'WATCH': 93, 'CTR_CHECK': 7}
Lead reason: {'PROFILE': 94, 'VISIBLE_LOW_CLICKS': 80, 'YOUNGER_PAGE': 61, 'HIGH_VISIBILITY': 32, 'LOW_SESSIONS': 13} 



,queue_rank,content_id,action,reason_codes,impressions_last_30d,clicks_last_30d,content_age_days
19,1,content_245531f14938,REVIEW_FIRST,VISIBLE_LOW_CLICKS|YOUNGER_PAGE,821,0,95
20,2,content_d6e535ef361f,REVIEW_FIRST,HIGH_VISIBILITY|LOW_SESSIONS,1092,0,112
21,3,content_4d7bba3dcb7c,REVIEW_FIRST,VISIBLE_LOW_CLICKS|YOUNGER_PAGE,2666,1,103
22,4,content_73df81553583,REVIEW_FIRST,VISIBLE_LOW_CLICKS|YOUNGER_PAGE,1547,0,103
23,5,content_fa3aad50d0be,REVIEW_FIRST,VISIBLE_LOW_CLICKS|YOUNGER_PAGE,519,0,98
24,6,content_b1b25cc9ee37,REVIEW_FIRST,VISIBLE_LOW_CLICKS|YOUNGER_PAGE,1171,0,98
25,7,content_33263b66aacf,REVIEW_FIRST,VISIBLE_LOW_CLICKS|YOUNGER_PAGE,779,0,98
26,8,content_12b8d8a8d62f,REVIEW_FIRST,VISIBLE_LOW_CLICKS|YOUNGER_PAGE,5692,1,98
27,9,content_af33d56338a4,WATCH,YOUNGER_PAGE|HIGH_VISIBILITY,459,0,95
28,10,content_c9c8fd0e7d30,REVIEW_FIRST,VISIBLE_LOW_CLICKS|YOUNGER_PAGE,577,0,119


1. **Each month, review each client's top-10 `REVIEW_FIRST` pages first** — highest decline risk with at least median traffic at stake.
2. **For `VISIBLE_LOW_CLICKS` pages, start with title and meta description** — impressions without clicks is the cheapest fix to test. Check the query the page is shown for before rewriting body content.
3. **For `YOUNGER_PAGE` pages, check whether rankings are still settling** before a full rewrite.
4. **Treat `PROFILE`-led and `WATCH` pages as lower priority** — re-check next month.
5. **Stop prioritising by staleness alone** — it showed no ranking signal on unseen clients.
6. **Measure the queue every month** (hit rate vs base rate) and retrain on the newest window; retire it if it falls to base rate two months running.

## 7. Artifacts the paper embeds

Charts go to `docs/img/` (served by GitHub Pages). One message per chart; the takeaway goes under each chart in the paper.

In [5]:
plt.rcParams.update({
    "svg.fonttype": "none", "font.family": "sans-serif",
    "font.sans-serif": ["Inter", "Helvetica", "Arial", "DejaVu Sans"], "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#6E7268",
    "axes.labelcolor": "#40453F", "xtick.color": "#40453F", "ytick.color": "#40453F",
    "figure.facecolor": "none", "axes.facecolor": "none",
})
INK, ACCENT, FLAG, MUTED, LINE = "#1A1D1B", "#2C3E63", "#93401F", "#A4A79D", "#DBDDD3"

# --- Figure 1: main result, precision@20 per fold with mean, base rate line ---
labels = {"random ranking": "Random ranking", "staleness only": "Staleness only (W02)",
          "W04 rule": "W04 hand-built rule", "momentum": "Impression momentum",
          "RF, W05 features (overlapping windows)": "RF, W05 features*",
          "RF, honest features": "Random forest", "LR, honest features": "Logistic regression (selected)"}
fig, ax = plt.subplots(figsize=(7.2, 3.9))
rng = np.random.default_rng(0)
for i, mdl in enumerate(order):
    vals = folds.loc[folds.model == mdl, "p20"].to_numpy()
    color = ACCENT if mdl == "LR, honest features" else (FLAG if mdl == "W04 rule" else MUTED)
    ax.scatter(vals, i + rng.uniform(-0.18, 0.18, len(vals)), s=14, color=color, alpha=0.45, lw=0)
    ax.plot([vals.mean()] * 2, [i - 0.32, i + 0.32], color=color, lw=4, solid_capstyle="round")
    ax.text(1.03, i, f"{vals.mean():.2f}", va="center", ha="left", fontsize=10.5, color=INK)
base = folds["base_rate"].mean()
ax.axvline(base, color=INK, ls=(0, (3, 3)), lw=1)
ax.text(base, len(order) - 0.35, f" base rate {base:.2f}", fontsize=9.5, color=INK, va="bottom")
ax.set_yticks(range(len(order))); ax.set_yticklabels([labels[o] for o in order])
ax.set_xlim(0.2, 1.0); ax.set_ylim(-0.6, len(order) - 0.1)
ax.set_xlabel("precision@20 on held-out clients (25 folds; bar = mean)")
ax.grid(axis="x", color=LINE, lw=0.6); ax.set_axisbelow(True)
fig.tight_layout(); fig.savefig(IMG / "fig1_results.svg"); plt.close(fig)

# --- Figure 2: how much a random split inflates the score (W05 features) ---
sc = pd.read_csv(OUT / "w06_split_compare.csv").groupby("split")["p20"].agg(["mean", "std"])
split_rows = {"Random row split": (sc.at["random", "mean"], sc.at["random", "std"]),
              "Client-grouped split": (sc.at["grouped", "mean"], sc.at["grouped", "std"]),
              "W05 single holdout": (0.600, None)}   # reported in w05_model.ipynb
fig, ax = plt.subplots(figsize=(7.2, 2.2))
for i, (lab, (mu, sd)) in enumerate(split_rows.items()):
    c = FLAG if i == 0 else ACCENT if i == 1 else MUTED
    ax.barh(i, mu, color=c, height=0.5)
    if sd: ax.errorbar(mu, i, xerr=sd, color=INK, capsize=3, lw=1)
    ax.text(0.02, i, f"{mu:.2f}" + (f" ± {sd:.2f}" if sd else " (one draw)"), va="center",
            color="white", fontsize=10.5, fontweight="bold")
ax.set_yticks(range(3)); ax.set_yticklabels(split_rows.keys()); ax.invert_yaxis()
ax.set_xlim(0, 1.05); ax.set_xlabel("precision@20, same W05 random forest")
fig.tight_layout(); fig.savefig(IMG / "fig2_split_gap.svg"); plt.close(fig)

# --- Figure 3: what the selected model leans on ---
nice = {"content_age_days": "Content age", "log_clicks_recent": "Clicks, recent month",
        "log_sessions_recent": "Sessions, recent month", "log_clicks_before": "Clicks, month before",
        "log_impressions_recent": "Impressions, recent month", "log_impressions_before": "Impressions, month before",
        "has_word_count": "Word count recorded", "log_sessions_before": "Sessions, month before",
        "main_intent_unknown": "Search intent unknown", "ctr_60d": "CTR, 60 days"}
top = coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(10)[::-1]
fig, ax = plt.subplots(figsize=(7.2, 3.6))
ax.barh([nice.get(k, k) for k in top.index], top.values,
        color=[FLAG if v > 0 else ACCENT for v in top.values], height=0.6)
ax.axvline(0, color=INK, lw=0.8)
ax.set_xlabel("standardised coefficient  (← lower decline risk | higher decline risk →)")
fig.tight_layout(); fig.savefig(IMG / "fig3_coefficients.svg"); plt.close(fig)

# --- Figure 4: what the live queue asks editors to do ---
lead = queue["reason_codes"].str.split("|").str[0].value_counts()
fig, ax = plt.subplots(figsize=(7.2, 2.6))
ax.barh(lead.index[::-1], lead.values[::-1], color=ACCENT, height=0.6)
for y_, v in enumerate(lead.values[::-1]):
    ax.text(v + 1, y_, str(v), va="center", fontsize=10.5, color=INK)
ax.set_xlabel(f"pages in the live queue by lead reason code (n = {len(queue)}, {queue['client_id'].nunique()} clients)")
fig.tight_layout(); fig.savefig(IMG / "fig4_reasons.svg"); plt.close(fig)

print(sorted(p.name for p in IMG.glob("fig*.svg")))

['fig1_results.svg', 'fig2_split_gap.svg', 'fig3_coefficients.svg', 'fig4_reasons.svg']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.